Usaremos como entrada:

../data/processed/ventas_validas.csv
../data/processed/ventas_invalidas.csv

cargaremos en PostgreSQL:

ventas_clean
ventas_error

INSTALL:
pip install pandas sqlalchemy psycopg2-binary python-dotenv

# Actividad 2.4 — Carga de datos a Base de Datos

## Objetivo

En este notebook implementaremos la etapa final del pipeline de datos: la carga controlada a una base de datos relacional PostgreSQL.

El flujo completo trabajado hasta ahora es:

1. Ingesta de datos
2. Limpieza y transformación
3. Validación estructural y semántica
4. Carga a base de datos ← etapa actual

En esta etapa cargaremos los registros válidos en la tabla `ventas_clean` y los registros rechazados en la tabla `ventas_error`.

## Importancia de la carga controlada

La carga de datos no consiste solo en insertar registros en una tabla. También implica:

- validar conexión a la base de datos
- verificar estructura de tablas
- respetar tipos de datos
- manejar errores
- registrar trazabilidad
- separar datos insertados y rechazados

Esto permite que el pipeline sea reproducible, auditable y confiable.

In [ ]:
import pandas as pd
import os
import logging
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

print("Bibliotecas cargadas...")

In [ ]:
os.makedirs("../logs", exist_ok=True)

logging.basicConfig(
    filename="../logs/load_database.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso de carga a base de datos")
print("Logger configurado correctamente")

## Conexión a PostgreSQL

La conexión se realizará usando variables de entorno desde el archivo `.env`.

Esto evita dejar credenciales directamente escritas en el notebook.

DB_HOST=TU_HOST

DB_PORT=TU_PORT

DB_NAME=TU_NAME

DB_USER=TU_USUARIO

DB_PASSWORD=TU_PASSWORD

In [ ]:
load_dotenv("../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

print("Cadena de conexión creada correctamente")

In [ ]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT current_database(), current_schema();"))
        print(result.fetchone())
        logging.info("Conexión a PostgreSQL exitosa")
except Exception as e:
    logging.error(f"Error de conexión a PostgreSQL: {e}")
    print("Error de conexión:", e)

## Creación/verificación de tablas

A continuación se crearán las tablas si no existen.

Usaremos:

- `ventas_clean`: registros válidos
- `ventas_error`: registros rechazados

In [ ]:
sql_create_tables = """
CREATE TABLE IF NOT EXISTS ventas_clean (
  id INT,
  fecha TEXT,
  producto TEXT,
  cantidad NUMERIC,
  precio NUMERIC,
  ciudad TEXT,
  total NUMERIC,
  estado_calidad TEXT,
  fuente TEXT
);

CREATE TABLE IF NOT EXISTS ventas_error (
  id INT,
  fecha TEXT,
  producto TEXT,
  cantidad TEXT,
  precio TEXT,
  ciudad TEXT,
  estado_calidad TEXT,
  fuente TEXT,
  observacion TEXT
);
"""

try:
    with engine.begin() as conn:
        conn.execute(text(sql_create_tables))
    logging.info("Tablas verificadas/creadas correctamente")
    print("Tablas verificadas o creadas correctamente")
except Exception as e:
    logging.error(f"Error creando tablas: {e}")
    print("Error creando tablas:", e)

## Lectura de archivos validados

Los archivos generados en la etapa anterior se encuentran en `data/processed`.

- `ventas_validas.csv`
- `ventas_invalidas.csv`

In [ ]:
ruta_validos = "../data/processed/ventas_validas.csv"
ruta_invalidos = "../data/processed/ventas_invalidas.csv"

df_validos = pd.read_csv(ruta_validos)
df_invalidos = pd.read_csv(ruta_invalidos)

print("Registros válidos:", len(df_validos))
print("Registros inválidos:", len(df_invalidos))

In [ ]:
df_validos.head()

In [ ]:
df_invalidos.head()

## Preparación de columnas para carga


Antes de insertar, seleccionaremos solo las columnas que existen en cada tabla destino.

In [ ]:
columnas_clean = [
    "id", "fecha", "producto", "cantidad", "precio",
    "ciudad", "total", "estado_calidad", "fuente"
]

columnas_error = [
    "id", "fecha", "producto", "cantidad", "precio",
    "ciudad", "estado_calidad", "fuente", "observacion"
]

df_validos_carga = df_validos[columnas_clean].copy()
df_invalidos_carga = df_invalidos[columnas_error].copy()

df_validos_carga.head()

In [ ]:
df_invalidos_carga.head()

## Validación previa a la carga

Validaremos que no existan datos críticos vacíos antes de insertar.

In [ ]:
def validar_dataframe_para_carga(df, columnas_obligatorias):
    errores = []

    for columna in columnas_obligatorias:
        if columna not in df.columns:
            errores.append(f"Falta la columna obligatoria: {columna}")
        elif df[columna].isnull().sum() > 0:
            errores.append(f"La columna {columna} contiene valores nulos")

    return errores


errores_validos = validar_dataframe_para_carga(
    df_validos_carga,
    ["id", "fecha", "producto", "cantidad", "precio", "ciudad", "estado_calidad"]
)

errores_invalidos = validar_dataframe_para_carga(
    df_invalidos_carga,
    ["id", "estado_calidad", "observacion"]
)

print("Errores en válidos:", errores_validos)
print("Errores en inválidos:", errores_invalidos)

## Carga de registros válidos

Insertaremos los registros válidos en la tabla `ventas_clean`.

In [ ]:
try:
    if not errores_validos:
        df_validos_carga.to_sql(
            "ventas_clean",
            engine,
            if_exists="append",
            index=False
        )
        logging.info(f"Registros válidos insertados: {len(df_validos_carga)}")
        print("Registros válidos insertados correctamente")
    else:
        logging.error(f"No se insertaron válidos por errores: {errores_validos}")
        print("No se insertaron válidos por errores previos")
except Exception as e:
    logging.error(f"Error insertando registros válidos: {e}")
    print("Error insertando registros válidos:", e)

## Carga de registros rechazados

Insertaremos los registros inválidos en la tabla `ventas_error`.

In [ ]:
try:
    if not errores_invalidos:
        df_invalidos_carga.to_sql(
            "ventas_error",
            engine,
            if_exists="append",
            index=False
        )
        logging.info(f"Registros inválidos insertados en tabla de error: {len(df_invalidos_carga)}")
        print("Registros inválidos insertados correctamente en ventas_error")
    else:
        logging.error(f"No se insertaron inválidos por errores: {errores_invalidos}")
        print("No se insertaron inválidos por errores previos")
except Exception as e:
    logging.error(f"Error insertando registros inválidos: {e}")
    print("Error insertando registros inválidos:", e)

## Verificación de carga

Consultaremos la base de datos para verificar cuántos registros quedaron cargados en cada tabla.

In [ ]:
with engine.connect() as conn:
    total_clean = conn.execute(text("SELECT COUNT(*) FROM ventas_clean;")).scalar()
    total_error = conn.execute(text("SELECT COUNT(*) FROM ventas_error;")).scalar()

print("Total registros en ventas_clean:", total_clean)
print("Total registros en ventas_error:", total_error)

In [ ]:
consulta = """
SELECT fuente, COUNT(*) AS total
FROM ventas_clean
GROUP BY fuente;
"""

pd.read_sql(consulta, engine)

In [ ]:
consulta_error = """
SELECT observacion, COUNT(*) AS total
FROM ventas_error
GROUP BY observacion
ORDER BY total DESC;
"""

pd.read_sql(consulta_error, engine)

## Trazabilidad del proceso

El proceso dejó registro en:

`../logs/load_database.log`

Ese archivo permite revisar:
- inicio del proceso
- conexión exitosa o fallida
- cantidad de registros insertados
- errores detectados durante la carga

In [ ]:
logging.info("Fin del proceso de carga a base de datos")
print("Proceso de carga finalizado")

## Reflexión final

Responde brevemente:

1. ¿Cómo se garantizó la trazabilidad del proceso de carga?
2. ¿Qué mecanismos permiten reproducir nuevamente este proceso?
3. ¿Qué errores podrían ocurrir durante una carga a base de datos?
4. ¿Por qué es importante separar registros válidos y rechazados?
5. ¿Cómo se relaciona esta etapa con la validación estructural y semántica?

## Conclusión

En este notebook implementamos la etapa de carga de datos a PostgreSQL, usando como entrada los registros previamente validados.

Esta etapa permite consolidar el pipeline, dejando los datos persistidos en una base relacional y disponibles para análisis, visualización o modelamiento posterior.